# 02 · Text to speech with SpeechT5

Now the other direction: from text to sound. **SpeechT5** does it in three steps:

1. An encoder-decoder transformer turns the text into a **mel spectrogram** (a picture of the sound: time on one axis, frequency on the other), frame by frame.
2. A **speaker embedding**, a vector of 512 numbers that describes a voice, tells the model *whose* voice to imitate.
3. A second network, the **vocoder** (HiFi-GAN), turns the spectrogram into a waveform with 16 000 samples per second.

**Download:** a few hundred MB with full-precision weights (the code prints the real size). The browser caches it. On a slow connection, set `DTYPE = "q8"` for a much smaller download with rougher sound.

In [ ]:
import time
import numpy as np
import lab

MODEL = "Xenova/speecht5_tts"
SPEAKER = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/speaker_embeddings.bin"
DTYPE = "fp32"        # "fp32" = full precision. "q8" = 8-bit weights: smaller download, rougher sound
DEVICE = "auto"       # "auto", "webgpu" (graphics card) or "wasm" (processor)
TEXT = "Hello! This sentence was spoken by a neural network running in my browser."

In [ ]:
tts = await lab.pipeline("text-to-speech", MODEL, device=DEVICE, dtype=DTYPE)

print(tts)
print("model files:", tts.size_mb, "MB, loaded in", tts.load_seconds, "s, running on", tts.device)

## Synthesize

The pipeline returns a dict with the waveform (`"audio"`, a NumPy array) and its sample rate (`"sampling_rate"`).

In [ ]:
start = time.time()
speech = await tts(TEXT, speaker_embeddings=SPEAKER)
seconds = time.time() - start

wave, sr = speech["audio"], speech["sampling_rate"]
duration = len(wave) / sr

print("samples:", wave.shape, wave.dtype)
print("sample rate:", sr, "Hz")
print("duration:", round(duration, 2), "s")
print("loudest sample:", round(float(np.abs(wave).max()), 3))
print(f"synthesis took {seconds:.1f} s: real-time factor {seconds / duration:.2f}")
lab.play(wave, sr)

## Look at the waveform

Sixteen thousand numbers per second are too many to draw one by one. The function below cuts the waveform into 600 columns and draws, for every column, a line from its smallest to its largest value. Words show up as bursts, pauses as flat stretches.

In [ ]:
def waveform_svg(wave, sr, columns=600, height=140):
    """An SVG drawing of a waveform: one vertical line (min to max) per column."""
    wave = np.asarray(wave, dtype=np.float64)
    usable = len(wave) - len(wave) % columns
    blocks = wave[:usable].reshape(columns, -1)
    low, high = blocks.min(axis=1), blocks.max(axis=1)
    scale = max(float(np.abs(wave).max()), 1e-9)
    mid = height / 2
    lines = "".join(
        f'<line x1="{i}" x2="{i}" y1="{mid - mid * h / scale:.1f}" y2="{mid - mid * l / scale:.1f}"/>'
        for i, (l, h) in enumerate(zip(low, high))
    )
    label = f"Waveform, {len(wave) / sr:.2f} seconds"
    return (
        f'<svg viewBox="0 0 {columns} {height}" width="{columns}" height="{height}" role="img" aria-label="{label}">'
        f'<g stroke="#2553c7" stroke-width="1">{lines}</g></svg>'
    )

lab.html(waveform_svg(wave, sr))

## How fast do people (and models) speak?

A quick sanity check of the output: count the words and divide by the duration. Ordinary speech has roughly two to three words per second.

In [ ]:
words = len(TEXT.split())
rate = words / duration
print(f"{words} words in {duration:.2f} s: {rate:.1f} words per second")

assert sr == 16000, "SpeechT5 produces 16 kHz audio"
assert wave.ndim == 1, "the waveform is mono: one number per sample"
assert 1.0 < rate < 6.0, "this does not look like speech: check TEXT and the model"

## Optional: does a machine understand the machine?

A round trip: let Whisper (notebook 01) transcribe the synthetic speech. This downloads Whisper tiny as well (roughly 40 to 150 MB), so it only runs if you set `ROUND_TRIP = True`.

In [ ]:
ROUND_TRIP = False

if ROUND_TRIP:
    asr = await lab.pipeline("automatic-speech-recognition", "Xenova/whisper-tiny.en", device=DEVICE)
    heard = await asr(wave)
    print("said: ", TEXT)
    print("heard:", heard["text"].strip())
else:
    print("Set ROUND_TRIP = True to run this cell.")

## Questions

1. Change `TEXT`: numbers, abbreviations, a foreign name, a question. Where does the model stumble, and why (think of what it saw during training)?
2. The speaker embedding has 512 numbers. What do you expect to hear if you use the embedding of another speaker? What if you average two embeddings?
3. Compare the real-time factor here with Whisper's in notebook 01. Which of the three steps do you think costs the most time?